# Sistem Pakar Fuzzy - Diagnosis Hama Padi 
## Wereng (Planthopper) vs Tikus (Rat) Detection System

Sistem ini menggunakan **Fuzzy Logic** untuk mendiagnosis hama padi berdasarkan:
- **Kerusakan Daun** (Leaf Damage)
- **Pola Kerusakan** (Damage Pattern)
- **Kerusakan Batang** (Stem Damage)
- **Waktu Serangan** (Attack Time)

---

## 1. Import Required Libraries

In [40]:
# Import NumPy untuk perhitungan numerik
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rcParams

# Set matplotlib untuk display yang lebih baik
rcParams['figure.figsize'] = (10, 6)
rcParams['font.size'] = 10

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


## 2. Define Membership Functions (Fungsi Keanggotaan)

In [41]:
# Triangular Membership Function (segitiga)
def trimf(x, a, b, c):
    """
    Fungsi keanggotaan segitiga
    a: nilai awal (kemiringan), b: puncak, c: nilai akhir (kemiringan)
    """
    if x <= a or x >= c: 
        return 0.0
    elif a < x <= b: 
        return (x - a) / (b - a)
    else: 
        return (c - x) / (c - b)

# Trapezoidal Membership Function (trapesium)
def trapmf(x, a, b, c, d):
    """
    Fungsi keanggotaan trapesium
    a: awal kemiringan, b: mulai flat, c: akhir flat, d: akhir kemiringan
    """
    if x <= a or x > d:  # FIXED: Changed x >= d to x > d
        return 0.0
    elif a < x <= b: 
        return (x - a) / (b - a)
    elif b < x <= c: 
        return 1.0
    else: 
        return (d - x) / (d - c)

print("✓ Membership functions defined!")

✓ Membership functions defined!


## 3. Implement Fuzzification for Input Variables (Fuzzifikasi Input)

In [42]:
# 3.1. Fuzzifikasi Kerusakan Daun (Leaf Damage)
def fuzzify_kerusakan_daun(nilai):
    """
    Mengkonversi nilai crisp (0-100%) menjadi fuzzy membership degrees
    - rendah: kerusakan sedikit
    - sedang: kerusakan sedang
    - tinggi: kerusakan parah
    """
    rendah = trapmf(nilai, 0, 0, 20, 40)
    sedang = trimf(nilai, 20, 50, 80)
    tinggi = trapmf(nilai, 60, 80, 100, 101)  # FIXED: Changed 100 to 101 for proper boundary
    return {"rendah": rendah, "sedang": sedang, "tinggi": tinggi}

# 3.2. Fuzzifikasi Pola Kerusakan (Damage Pattern)
def fuzzify_pola_kerusakan(nilai):
    """
    - merata: tersebar merata di seluruh daun (ciri Wereng)
    - campuran: tersebar tidak merata
    - spot: bintik-bintik terlokalisir (ciri Tikus)
    """
    merata = trapmf(nilai, 0, 0, 20, 40)
    campuran = trimf(nilai, 20, 50, 80)
    spot = trapmf(nilai, 60, 80, 100, 101)  # FIXED: Changed 100 to 101 for proper boundary
    return {"merata": merata, "campuran": campuran, "spot": spot}

# 3.3. Fuzzifikasi Kerusakan Batang (Stem Damage)
def fuzzify_kerusakan_batang(nilai):
    """
    - layu: batang layu/mengecil (ciri Wereng)
    - lubang: ada lubang di batang
    - potong: batang terpotong/putus (ciri Tikus)
    """
    layu = trapmf(nilai, 0, 0, 20, 40)
    lubang = trimf(nilai, 25, 50, 75)
    potong = trapmf(nilai, 60, 80, 100, 101)  # FIXED: Changed 100 to 101 for proper boundary
    return {"layu": layu, "lubang": lubang, "potong": potong}

# 3.4. Fuzzifikasi Waktu Serangan (Attack Time)
def fuzzify_waktu_serangan(nilai):
    """
    - siang: serangan siang hari (ciri Wereng)
    - campuran: aktif siang dan malam
    - malam: serangan malam hari (ciri Tikus)
    """
    siang = trapmf(nilai, 0, 0, 20, 40)
    campuran = trimf(nilai, 20, 50, 80)
    malam = trapmf(nilai, 60, 80, 100, 101)  # FIXED: Changed 100 to 101 for proper boundary
    return {"siang": siang, "campuran": campuran, "malam": malam}

print("✓ Fuzzification functions created!")

✓ Fuzzification functions created!


## 4. Create Rule Base for Pest Diagnosis (Basis Aturan)

In [43]:
def apply_rules(daun, pola, batang, waktu):
    """
    Menerapkan aturan fuzzy untuk mendiagnosis hama padi
    Menggunakan operator MIN untuk logika AND
    
    Struktur rule: (nama_hama, tingkat_keparahan, derajat_kepastian)
    """
    rules = []
    
    # =====================================================
    # WERENG (Planthopper) RULES - Ciri khas: merata, layu, siang
    # =====================================================
    
    # Rule 1-3: Tinggi Wereng
    rules.append(("wereng", "tinggi", min(daun["tinggi"], pola["merata"], batang["layu"], waktu["siang"])))
    rules.append(("wereng", "tinggi", min(daun["tinggi"], pola["merata"], batang["layu"], waktu["campuran"])))
    rules.append(("wereng", "tinggi", min(daun["tinggi"], pola["merata"], batang["layu"])))
    
    # Rule 4-8: Sedang Wereng
    rules.append(("wereng", "sedang", min(daun["sedang"], pola["merata"], batang["layu"])))
    rules.append(("wereng", "sedang", min(daun["tinggi"], pola["campuran"], batang["layu"])))
    rules.append(("wereng", "sedang", min(pola["merata"], batang["layu"], waktu["siang"])))
    rules.append(("wereng", "sedang", min(daun["sedang"], waktu["siang"], pola["merata"])))
    rules.append(("wereng", "sedang", min(daun["sedang"], pola["campuran"], waktu["siang"])))
    
    # Rule 9: Rendah Wereng
    rules.append(("wereng", "rendah", min(daun["rendah"], pola["merata"], batang["layu"])))
    
    # =====================================================
    # TIKUS (Rat) RULES - Ciri khas: spot, potong, malam
    # =====================================================
    
    # Rule 10-13: Tinggi Tikus
    rules.append(("tikus", "tinggi", min(batang["potong"], pola["spot"], waktu["malam"])))
    rules.append(("tikus", "tinggi", min(batang["potong"], pola["spot"])))
    rules.append(("tikus", "tinggi", min(daun["rendah"], batang["potong"], waktu["malam"])))
    rules.append(("tikus", "tinggi", min(daun["rendah"], batang["potong"], pola["spot"])))
    
    # Rule 14-23: Sedang Tikus
    rules.append(("tikus", "sedang", min(batang["lubang"], pola["spot"], waktu["malam"])))
    rules.append(("tikus", "sedang", min(daun["rendah"], pola["spot"], batang["potong"])))
    rules.append(("tikus", "sedang", min(batang["lubang"], pola["spot"])))
    rules.append(("tikus", "sedang", min(pola["spot"], waktu["malam"], daun["sedang"])))
    rules.append(("tikus", "sedang", min(batang["lubang"], waktu["campuran"], pola["spot"])))
    rules.append(("tikus", "sedang", min(daun["sedang"], pola["spot"], waktu["malam"])))
    
    # Rule 24-25: Rendah Tikus
    rules.append(("tikus", "rendah", min(batang["lubang"], waktu["campuran"])))
    rules.append(("tikus", "rendah", min(daun["rendah"], pola["campuran"], waktu["malam"])))
    
    return rules

print("✓ Rule base system initialized!")

✓ Rule base system initialized!


## 5. Implement Defuzzification Process (Defuzzifikasi)

In [44]:
def defuzzify(rules, hama):
    """
    Mengubah output fuzzy agregat menjadi nilai crisp (0-100)
    Menggunakan metode Center of Gravity (COG)
    
    1. Ambil semua rule untuk hama tertentu
    2. Agregasi dengan MAX (union)
    3. Hitung center of gravity
    """
    x = np.linspace(0, 100, 1000)  # 1000 titik sample
    aggregated = np.zeros_like(x)
    
    # Iterasi setiap rule untuk hama yang ditargetkan
    for (target_hama, level, alpha) in rules:
        if target_hama != hama or alpha == 0:
            continue
        
        # Pilih membership function sesuai level
        if level == "rendah":
            mf = np.array([trapmf(xi, 0, 0, 20, 40) for xi in x])
        elif level == "sedang":
            mf = np.array([trimf(xi, 20, 50, 80) for xi in x])
        else:  # tinggi
            mf = np.array([trapmf(xi, 60, 80, 100, 100) for xi in x])
        
        # Clip output dengan alpha (derajat rule)
        clipped = np.minimum(alpha, mf)
        
        # Agregasi dengan MAX
        aggregated = np.maximum(aggregated, clipped)
    
    # Center of Gravity calculation
    if np.sum(aggregated) == 0:
        return 0.0
    
    cogValue = np.sum(x * aggregated) / np.sum(aggregated)
    return round(cogValue, 2)

print("✓ Defuzzification function ready!")

✓ Defuzzification function ready!


## 6. Build Diagnosis Function (Fungsi Diagnosis)

In [45]:
def diagnosa(kerusakan_daun, pola_kerusakan, kerusakan_batang, waktu_serangan):
    """
    Fungsi utama untuk mendiagnosis hama padi
    
    Flow:
    1. Fuzzifikasi (Crisp → Fuzzy)
    2. Terapkan rules
    3. Defuzzifikasi (Fuzzy → Crisp)
    4. Tentukan diagnosis & kepercayaan
    
    Input (0-100):
    - kerusakan_daun: persentase kerusakan daun
    - pola_kerusakan: tipe pola kerusakan
    - kerusakan_batang: persentase kerusakan batang
    - waktu_serangan: waktu serangan
    
    Output: dict dengan hasil diagnosis
    """
    
    # STEP 1: Fuzzifikasi
    daun = fuzzify_kerusakan_daun(kerusakan_daun)
    pola = fuzzify_pola_kerusakan(pola_kerusakan)
    batang = fuzzify_kerusakan_batang(kerusakan_batang)
    waktu = fuzzify_waktu_serangan(waktu_serangan)
    
    # STEP 2: Terapkan Rules
    rules = apply_rules(daun, pola, batang, waktu)
    
    # STEP 3: Defuzzifikasi
    skor_wereng = defuzzify(rules, "wereng")
    skor_tikus = defuzzify(rules, "tikus")
    
    # STEP 4: Hitung persentase kepercayaan
    total = skor_wereng + skor_tikus
    if total == 0:
        pct_wereng = pct_tikus = 50.0
    else:
        pct_wereng = round((skor_wereng / total) * 100, 1)
        pct_tikus = round((skor_tikus / total) * 100, 1)
    
    # STEP 5: Tentukan diagnosis akhir
    if pct_wereng >= pct_tikus:
        diagnosis = "🐛 WERENG (Planthopper)"
        confidence = pct_wereng
        penanganan = [
            "Semprotkan insektisida"
        ]
    else:
        diagnosis = "🐭 TIKUS (Rat)"
        confidence = pct_tikus
        penanganan = [
            "Semprotkan rodentisida",
            "Pasang perangkap tikus",
            "Karantina area yang terjangkit"
        ]
    
    return {
        "diagnosis": diagnosis,
        "confidence": confidence,
        "skor_wereng": skor_wereng,
        "skor_tikus": skor_tikus,
        "pct_wereng": pct_wereng,
        "pct_tikus": pct_tikus,
        "penanganan": penanganan,
        "input": {
            "kerusakan_daun": kerusakan_daun,
            "pola_kerusakan": pola_kerusakan,
            "kerusakan_batang": kerusakan_batang,
            "waktu_serangan": waktu_serangan
        },
        "fuzzy_values": {
            "daun": daun,
            "pola": pola,
            "batang": batang,
            "waktu": waktu
        }
    }

print("✓ Diagnosis function ready!")

✓ Diagnosis function ready!


## 7. Results Display & Visualization

In [46]:
def print_hasil(hasil):
    """
    Menampilkan hasil diagnosis dengan format yang rapi
    """
    print("\n" + "="*70)
    print("  SISTEM PAKAR FUZZY - DIAGNOSIS HAMA PADI ")
    print("="*70)
    
    inp = hasil["input"]
    fv = hasil["fuzzy_values"]
    
    def label_pola(v): 
        return "Merata" if v<=33 else "Campuran" if v<=66 else "Spot"
    
    def label_batang(v): 
        return "Layu" if v<=33 else "Lubang" if v<=66 else "Potong"
    
    def label_waktu(v): 
        return "Siang" if v<=33 else "Campuran" if v<=66 else "Malam"
    
    # INPUT
    print(f"\n INPUT DATA:")
    print(f"  • Kerusakan Daun  : {inp['kerusakan_daun']}%")
    print(f"  • Pola Kerusakan  : {inp['pola_kerusakan']} ({label_pola(inp['pola_kerusakan'])})")
    print(f"  • Kerusakan Batang: {inp['kerusakan_batang']} ({label_batang(inp['kerusakan_batang'])})")
    print(f"  • Waktu Serangan  : {inp['waktu_serangan']} ({label_waktu(inp['waktu_serangan'])})")
    
    # FUZZY VALUES
    print(f"\n NILAI FUZZY (Derajat Keanggotaan):")
    d, f = fv["daun"], fv["pola"]
    b, w = fv["batang"], fv["waktu"]
    
    print(f"  Daun      → Rendah: {d['rendah']:.3f} | Sedang: {d['sedang']:.3f} | Tinggi: {d['tinggi']:.3f}")
    print(f"  Pola      → Merata: {f['merata']:.3f} | Campuran: {f['campuran']:.3f} | Spot: {f['spot']:.3f}")
    print(f"  Batang    → Layu: {b['layu']:.3f} | Lubang: {b['lubang']:.3f} | Potong: {b['potong']:.3f}")
    print(f"  Waktu     → Siang: {w['siang']:.3f} | Campuran: {w['campuran']:.3f} | Malam: {w['malam']:.3f}")
    
    # DERAJAT OUTPUT
    print(f"\n DERAJAT OUTPUT:")
    print(f"  • Derajat Wereng : {hasil['skor_wereng']:.2f}")
    print(f"  • Derajat Tikus  : {hasil['skor_tikus']:.2f}")
    
    # DIAGNOSIS
    print(f"\n DIAGNOSIS AKHIR:")
    print(f"  {hasil['diagnosis']}")
    print(f"  Tingkat Kepercayaan: {hasil['confidence']}%")
    print(f"  Wereng: {hasil['pct_wereng']}% | Tikus: {hasil['pct_tikus']}%")
    
    # REKOMENDASI
    print(f"\n REKOMENDASI PENANGANAN:")
    for i, p in enumerate(hasil["penanganan"], 1):
        print(f"  {i}. {p}")
    
    print("="*70 + "\n")

print("✓ Display function ready!")

✓ Display function ready!


## 8. Test Cases & System Validation

In [47]:
# ========== TEST CASE 1: Strong WERENG indicators ==========
hasil1 = diagnosa(80, 10, 15, 15)
print_hasil(hasil1)

# ========== TEST CASE 2: Strong TIKUS indicators ==========
hasil2 = diagnosa(15, 90, 90, 90)
print_hasil(hasil2)

# ========== TEST CASE 3: Ambiguous/Mixed ==========
hasil3 = diagnosa(50, 50, 50, 50)
print_hasil(hasil3)

# ========== TEST CASE 4: Mixed signals leaning TIKUS ==========
hasil4 = diagnosa(30, 70, 55, 80)
print_hasil(hasil4)

print("✓ Semua test case selesai!")


  SISTEM PAKAR FUZZY - DIAGNOSIS HAMA PADI 

 INPUT DATA:
  • Kerusakan Daun  : 80%
  • Pola Kerusakan  : 10 (Merata)
  • Kerusakan Batang: 15 (Layu)
  • Waktu Serangan  : 15 (Siang)

 NILAI FUZZY (Derajat Keanggotaan):
  Daun      → Rendah: 0.000 | Sedang: 0.000 | Tinggi: 1.000
  Pola      → Merata: 1.000 | Campuran: 0.000 | Spot: 0.000
  Batang    → Layu: 1.000 | Lubang: 0.000 | Potong: 0.000
  Waktu     → Siang: 1.000 | Campuran: 0.000 | Malam: 0.000

 DERAJAT OUTPUT:
  • Derajat Wereng : 67.10
  • Derajat Tikus  : 0.00

 DIAGNOSIS AKHIR:
  🐛 WERENG (Planthopper)
  Tingkat Kepercayaan: 100.0%
  Wereng: 100.0% | Tikus: 0.0%

 REKOMENDASI PENANGANAN:
  1. Semprotkan insektisida


  SISTEM PAKAR FUZZY - DIAGNOSIS HAMA PADI 

 INPUT DATA:
  • Kerusakan Daun  : 15%
  • Pola Kerusakan  : 90 (Spot)
  • Kerusakan Batang: 90 (Potong)
  • Waktu Serangan  : 90 (Malam)

 NILAI FUZZY (Derajat Keanggotaan):
  Daun      → Rendah: 1.000 | Sedang: 0.000 | Tinggi: 0.000
  Pola      → Merata: 0.000 |